# HP-3 · Sinh dữ liệu thời lượng đọc

**Mục tiêu.** Với mỗi câu tiếng Việt, cho VieNeu-TTS đọc thật rồi đo số giây.
Kết quả là bộ ba *(văn bản, giọng, thời lượng)* — dữ liệu để huấn luyện mô hình
dự đoán thời lượng ở notebook `02`.

**Vì sao cần.** Lồng tiếng phải vừa khung thời gian của câu thoại gốc. Muốn biết
trước câu nào đọc không kịp thì phải đoán được câu đó đọc mất bao lâu, *trước
khi* đọc. Đó là MT-3 trong đề cương (mục tiêu MAE < 200 ms).

**Đầu vào.** File văn bản, mỗi dòng một câu (`data/seed_vi.txt`).
**Đầu ra.** `data/duration/samples.csv` với 4 cột `text, voice, style, seconds`.

**Lưu ý**
- Cần cài nhóm TTS: thêm `--extra tts` vào lệnh `uv sync` đang dùng.
- Số giây đo *sau khi cắt lặng đầu/cuối*, đúng như lúc lồng tiếng thật.
- Chạy lại thì tự bỏ qua câu đã đo, nên ngắt giữa chừng không mất gì.
- Bộ mẫu 90 câu chỉ để chạy thử đường ống. Dữ liệu thật cần 30.000–50.000 câu.

In [ ]:
# Chạy được dù mở notebook từ thư mục nào: đi ngược lên tới thư mục có backend/
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "backend").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Gốc dự án:", ROOT)

In [ ]:
from backend.core import dubbing

ok, why = dubbing.available(deep=True)
print("TTS dùng được:", ok)
if not ok:
    print("Lý do:", why)
    print("Cài thêm: thêm --extra tts vào lệnh uv sync đang dùng.")

## Cấu hình

Phải chốt tên giọng **ngay tại đây**, trước khi đối chiếu với phần đã đo. CSV ghi
tên giọng thật, nên nếu để `VOICE` rỗng rồi mới lấy giọng ở bước đo thì không câu
nào khớp với dòng cũ — cả bộ bị đo lại từ đầu, mất hàng giờ TTS mà không báo gì.

In [ ]:
INPUT = ROOT / "training" / "data" / "seed_vi.txt"
OUTPUT = ROOT / "data" / "duration" / "samples.csv"
STYLE = "tu_nhien"
VOICE = ""      # trống = lấy giọng đầu tiên của engine
LIMIT = 0       # 0 = đo hết; đặt số nhỏ để chạy thử

if not VOICE:
    voices = dubbing.list_voices()
    VOICE = voices[0]["id"] if voices else ""

print("Phong cách có sẵn:", [s["id"] for s in dubbing.STYLES])
print("Giọng dùng để đo :", VOICE or "(mặc định của engine)")

## Đọc câu và bỏ qua phần đã đo

In [ ]:
import csv


def read_sentences(path):
    """Mỗi dòng một câu; bỏ dòng trống, dòng chú thích (#) và câu trùng."""
    seen, out = set(), []
    for line in path.read_text(encoding="utf-8").splitlines():
        s = " ".join(line.split())
        if s and not s.startswith("#") and s not in seen:
            seen.add(s)
            out.append(s)
    return out


def already_done(path):
    """Bộ ba đã đo rồi -> lần chạy sau không đo lại."""
    if not path.exists():
        return set()
    with path.open(encoding="utf-8", newline="") as fh:
        return {(r["text"], r["voice"], r["style"]) for r in csv.DictReader(fh)}


sentences = read_sentences(INPUT)
if LIMIT:
    sentences = sentences[:LIMIT]
done = already_done(OUTPUT)
todo = [s for s in sentences if (s, VOICE, STYLE) not in done]
print(f"{len(sentences)} câu · {len(done)} đã đo · còn {len(todo)}")

## Đo thời lượng\n\nMỗi câu là một lần gọi TTS nên bước này lâu. Ô này ghi thẳng ra CSV sau từng câu.

In [ ]:
import time

import numpy as np

engine = dubbing.get_engine()
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
new_file = not OUTPUT.exists()
t0 = time.time()

with OUTPUT.open("a", encoding="utf-8", newline="") as fh:
    writer = csv.writer(fh)
    if new_file:
        writer.writerow(["text", "voice", "style", "seconds"])
    for i, text in enumerate(todo, 1):
        try:
            kwargs = {"style": STYLE}
            if VOICE:
                kwargs["voice"] = VOICE
            audio = np.asarray(engine.infer(text, **kwargs), dtype=np.float32).reshape(-1)
            audio, _ = dubbing._trim_silence(audio)
            seconds = audio.size / dubbing.SR
        except Exception as exc:
            print(f"  [{i}/{len(todo)}] lỗi, bỏ qua: {exc}")
            continue
        # TTS đôi khi không gặp điểm kết thúc và sinh tới trần 24 giây —
        # mẫu đó sai, đưa vào huấn luyện thì làm hỏng mô hình.
        if dubbing._runaway(seconds, text):
            print(f"  [{i}/{len(todo)}] chạy loạn ({seconds:.1f}s), bỏ: {text[:40]}")
            continue
        writer.writerow([text, VOICE, STYLE, f"{seconds:.3f}"])
        fh.flush()
        if i % 10 == 0 or i == len(todo):
            rate = (time.time() - t0) / i
            print(f"  [{i}/{len(todo)}] {seconds:5.2f}s · ~{rate:.1f}s/câu, "
                  f"còn ~{rate * (len(todo) - i) / 60:.0f} phút")

print("Xong ->", OUTPUT)

## Xem nhanh dữ liệu vừa sinh

In [ ]:
with OUTPUT.open(encoding="utf-8", newline="") as fh:
    rows = list(csv.DictReader(fh))

print(f"{len(rows)} mẫu trong {OUTPUT.name}")
if rows:
    secs = [float(r["seconds"]) for r in rows]
    chars = [len(r["text"]) for r in rows]
    print(f"  thời lượng: {min(secs):.2f}–{max(secs):.2f}s, trung bình {sum(secs)/len(secs):.2f}s")
    print(f"  độ dài câu: {min(chars)}–{max(chars)} ký tự")
    print("\nVài dòng đầu:")
    for r in rows[:5]:
        print(f'  {float(r["seconds"]):5.2f}s  {r["text"][:60]}')

Đủ dữ liệu rồi thì sang **`02_huan_luyen_duration.ipynb`** để huấn luyện.